In [ ]:
# ==========================================================
# 🌟 THE FINAL GITHUB-READY MASTER CELL (LOCAL PATHS & SECURE)
# ==========================================================
import warnings
import logging
import os
import getpass
from pathlib import Path

# 1. كتم التحذيرات
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.ERROR)

print("⏳ جاري تشغيل النسخة الذكية (بدون جوجل درايف)...")

# 2. تسطيب المكتبات
from IPython.utils import io
with io.capture_output() as captured:
    !pip install -q chromadb easyocr sentence-transformers duckduckgo-search google-genai ultralytics

import cv2
import chromadb
from chromadb.utils import embedding_functions
from google import genai
import easyocr
from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Markdown, Image

# ==========================================================
# 📁 مسارات المشروع المحلية (Local Paths)
# ==========================================================
# المسار الأساسي هو فولدر "ai shopping" بجوار ملف النوت بوك
BASE_DIR = Path.cwd() / "ai shopping"

YOLO_MODEL_PATH = BASE_DIR / "models" / "best.pt"
DRIVE_OCR_MODELS = BASE_DIR / "models" / "easyocr_weights"
CHROMA_DB_PATH = BASE_DIR / "chroma_db"

# التأكد من إنشاء المجلدات أوتوماتيكياً لتجنب أي أخطاء عند المستخدمين
YOLO_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
DRIVE_OCR_MODELS.mkdir(parents=True, exist_ok=True)
CHROMA_DB_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# 🧠 إعداد ChromaDB بدقة متناهية
# ==========================================================
with io.capture_output() as captured:
    chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="intfloat/multilingual-e5-base"
    )
    
    try:
        chroma_client.delete_collection(name="strict_shopping_assistant")
    except:
        pass
        
    collection = chroma_client.create_collection(
        name="strict_shopping_assistant", 
        embedding_function=embedding_func,
        metadata={"hnsw:space": "cosine"} 
    )

sample_products = [
    {"id": "1", "name": "شيبسي كباب", "price": "10 جنيه", "description": "رقائق بطاطس مقرمشة بطعم الكباب.", "keywords": "شيبسي كباب شيبس بطاطس مقرمشات chips kebab"},
    {"id": "2", "name": "عصير جهينة مانجو", "price": "15 جنيه", "description": "عصير طبيعي بدون مواد حافظة.", "keywords": "عصير جهينة مانجو شرب juhayna mango juice"},
    {"id": "3", "name": "بسكويت أوريو", "price": "5 جنيه", "description": "بسكويت بالشوكولاتة محشو بالكريمة.", "keywords": "بسكويت اوريو بسكوت شوكولاته oreo biscuit"},
    {"id": "4", "name": "بيبسي دايت", "price": "12 جنيه", "description": "مشروب غازي خالي من السكر.", "keywords": "بيبسي دايت كانز مشروب غازي pepsi diet can"}
]

with io.capture_output() as captured:
    collection.add(
        ids=[p["id"] for p in sample_products],
        documents=[p["keywords"] for p in sample_products],
        metadatas=[{"name": p["name"], "price": p["price"], "description": p["description"]} for p in sample_products]
    )

# ==========================================================
# 🛠️ المحركات مع الفلتر الصارم
# ==========================================================

class ProductDetector:
    def __init__(self):
        if YOLO_MODEL_PATH.exists():
            with io.capture_output() as captured:
                self.model = YOLO(str(YOLO_MODEL_PATH))
        else:
            self.model = None

    def detect_and_crop(self, image_path):
        img = cv2.imread(image_path)
        if self.model:
            results = self.model(img, conf=0.25, verbose=False) 
            if len(results[0].boxes) > 0:
                box = results[0].boxes[0].xyxy[0].cpu().numpy().astype(int)
                return img[box[1]:box[3], box[0]:box[2]]
        return img 

class UltimateOCR:
    def __init__(self):
        with io.capture_output() as captured:
            self.reader = easyocr.Reader(['ar', 'en'], gpu=True, model_storage_directory=str(DRIVE_OCR_MODELS), verbose=False)

    def extract_text(self, img):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore") 
            detections = self.reader.readtext(img, text_threshold=0.1, low_text=0.2)
        valid_texts = [str(text).strip() for bbox, text, conf in detections if len(str(text).strip()) > 1]
        return " ".join(valid_texts) if valid_texts else ""

class StrictRetriever:
    def __init__(self, collection):
        self.collection = collection

    def get_product_info(self, ocr_text):
        if len(ocr_text.strip()) < 2: 
            return "المنتج غير مسجل في قاعدة البيانات."
        
        results = self.collection.query(
            query_texts=[f"query: {ocr_text}"],
            n_results=1
        )
        
        if results['distances'] and len(results['distances'][0]) > 0:
            distance = results['distances'][0][0]
            meta = results['metadatas'][0][0]
            doc = results['documents'][0][0]
            
            ocr_words = set(ocr_text.lower().split())
            doc_words = set(doc.lower().split())
            common = ocr_words.intersection(doc_words)
            
            if distance < 0.15 or len(common) > 0:
                return f"الاسم: {meta['name']} | السعر: {meta['price']} | الوصف: {meta['description']}"
            
        return "المنتج غير مسجل في قاعدة البيانات المحلية."

class GeminiAssistant:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)
        self.model_id = "gemini-3.5-flash"

    def generate_answer(self, ocr_text, product_info, user_query):
        prompt = f"""أنت مساعد تسوق ذكي.
معلومات من قاعدة بيانات المتجر: {product_info}
النص الحقيقي المكتوب على الصورة (قراءة OCR): "{ocr_text}"
سؤال المستخدم: {user_query}
أجب على السؤال بدقة شديدة بناءً على البيانات المتوفرة وبإيجاز شديد:"""
        try:
            response = self.client.models.generate_content(
                model=self.model_id,
                contents=prompt,
            )
            return response.text.strip()
        except Exception as e:
            return f"❌ خطأ Gemini: {str(e)}"

# ==========================================================
# 🚀 إدخال الـ API بشكل آمن وتجهيز النظام
# ==========================================================
print("\n" + "="*50)
if 'YOUR_GEMINI_API_KEY' not in locals() or not YOUR_GEMINI_API_KEY:
    YOUR_GEMINI_API_KEY = getpass.getpass("🔑 يرجى إدخال مفتاح Gemini API الخاص بك والصقه هنا ثم اضغط Enter: ")
print("="*50)

detector = ProductDetector()
ocr_engine = UltimateOCR()
retriever = StrictRetriever(collection)
llm_engine = GeminiAssistant(YOUR_GEMINI_API_KEY)

print("✅ النظام جاهز للاختبار محلياً!")

# ==========================================================
# 🖥️ واجهة التشغيل
# ==========================================================
print("\n" + "🌟 "*15)
uploaded = files.upload()

if uploaded:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        image_path = list(uploaded.keys())[0]
        display(Image(filename=image_path, width=250))
        
        user_query = input("❓ اكتب سؤالك عن المنتج: ")
        if not user_query.strip(): user_query = "ما هي تفاصيل هذا المنتج؟"
        
        print("\n⏳ جاري المعالجة بدقة صارمة...")
        
        try:
            with io.capture_output() as captured:
                cropped_img = detector.detect_and_crop(image_path)
                ocr_text = ocr_engine.extract_text(cropped_img)
                product_info = retriever.get_product_info(ocr_text)
            
            print(f"\n[🔍 System Output]: {product_info}\n")
            
            final_answer = llm_engine.generate_answer(ocr_text, product_info, user_query)
            
            print("🌟 "*15)
            display(Markdown(f"### {final_answer}"))
            print("🌟 "*15)
            
        except Exception as e:
            print(f"\n❌ حدث خطأ: {str(e)}")
        finally:
            if os.path.exists(image_path): os.remove(image_path)

In [ ]:
# ==========================================================
# 🎨 GRADIO INTERFACE CELL (PROFESSIONAL & CLEAN)
# ==========================================================
import gradio as gr
import PIL.Image
import os

def shopping_assistant_ui(input_image, query):
    if input_image is None:
        return "❌ الرجاء رفع صورة المنتج أولاً."
        
    if not query.strip():
        query = "ما هي تفاصيل هذا المنتج؟"

    # حفظ الصورة مؤقتاً لتمريرها للمحركات
    temp_path = "temp_gradio_img.jpg"
    input_image.save(temp_path)

    try:
        # 1. Detection & Cropping using YOLO
        print("⏳ جاري تحديد وقص المنتج...")
        cropped = detector.detect_and_crop(temp_path)

        # 2. OCR Extraction using EasyOCR
        print("⏳ جاري استخراج النصوص من الصورة...")
        ocr_text = ocr_engine.extract_text(cropped)

        # 3. Vector Search using ChromaDB (Strict Matching)
        print("⏳ جاري البحث في قاعدة البيانات بدقة...")
        product_info = retriever.get_product_info(ocr_text)

        # 4. Gemini Reasoning
        print("⏳ جاري توليد الإجابة الذكية...")
        answer = llm_engine.generate_answer(ocr_text, product_info, query)

        return answer
    except Exception as e:
        return f"❌ حدث خطأ أثناء المعالجة: {str(e)}"
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

# ==========================================================
# 🖥️ إعداد وتصميم الواجهة (UI Setup)
# ==========================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<center><h1>🛒 المساعد الذكي للتسوق (AI Shopping Assistant)</h1></center>")
    gr.Markdown("<center>يعتمد هذا النظام على YOLO لاكتشاف المنتج، EasyOCR لقراءة النصوص، ChromaDB للبحث الدقيق المانع للهلوسة، و Gemini 3.5 Flash للتفكير المنطقي.</center>")

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="صورة المنتج (Image Input)")
            query_input = gr.Textbox(
                label="سؤالك عن المنتج",
                placeholder="مثلاً: هل هذا المشروب مناسب لمريض السكر؟",
                lines=2
            )
            submit_btn = gr.Button("🔍 تحليل المنتج والإجابة", variant="primary")

        with gr.Column():
            # استخدام Markdown لعرض الإجابة بشكل منسق وشيك
            output_text = gr.Markdown(label="إجابة المساعد الذكي")

    submit_btn.click(
        fn=shopping_assistant_ui,
        inputs=[img_input, query_input],
        outputs=output_text
    )

print("🚀 جاري إنشاء الرابط العام للواجهة (Gradio Share Link)...")
# إغلاق الـ Debugging عشان الـ Terminal يفضل نظيف قدام اللجنة
demo.launch(share=True, debug=False)